# HIGGS Benchmark on Colab or Kaggle — Progress-Visible, Resource-Safe Run

This notebook validates the complete 11-million-row raw dataset, then defaults to a **quick training benchmark on the prepared 250,000-row sample** so resource-limited runtimes can produce results instead of silently loading the full CSV. The raw 11-million-row file remains available. The quick run is a real training/evaluation run but must not be described as full-data training.


## Step 1A — Imports and project settings

Run once at the beginning. This cell only sets the repository path and URL. It also records the runtime assumptions needed for later cells and keeps the workflow reproducible on both Colab and Kaggle.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/CosmicMagnetar/higgs-event-lab.git'
cwd = Path.cwd().resolve()
candidates = []
if os.environ.get("HIGGS_REPO_ROOT"):
    candidates.append(Path(os.environ["HIGGS_REPO_ROOT"]).expanduser().resolve())
candidates.extend([cwd, cwd / "higgs-event-lab", *cwd.parents])

def is_higgs_repo(path):
    return (path / "src" / "higgs_event_lab").is_dir() and (path / "pyproject.toml").is_file()

root = next((path for path in dict.fromkeys(candidates) if is_higgs_repo(path)), None)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
if root is None:
    root = cwd / "higgs-event-lab"
    suffix = 1
    while root.exists() and not is_higgs_repo(root):
        if root.is_dir() and not any(root.iterdir()):
            break
        suffix += 1
        root = cwd / f"higgs-event-lab-{suffix}"
print("Repository path:", root)


## Step 1B — Clone the repository if needed

If the source folder is already present, this skips cloning. The cell is safe to rerun after an interrupted session and prevents accidental use of an incomplete directory.


In [ ]:
if not is_higgs_repo(root):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
if not is_higgs_repo(root):
    raise FileNotFoundError(f"Repository clone is incomplete: {root}")

os.environ["HIGGS_REPO_ROOT"] = str(root)
os.chdir(root)
print("Repository ready:", root)


## Step 1C — Install project requirements

Run this after Step 1B. Package installation can take a few minutes. Installing the pinned project dependency set ensures that the repository modules and notebook commands use the same environment.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")], check=True)
src_dir = root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
print("Dependencies installed; project source path added.")


## Step 2A — Download the complete HIGGS dataset

The URL below points to the official UCI archive containing the complete approximately 11-million-row HIGGS dataset. This notebook uses that complete raw file for EDA, reconciliation and model training. No derived 250k or 1M training sample is used.


In [ ]:
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab")).resolve()
raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
raw_path.parent.mkdir(parents=True, exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/280/higgs.zip"
zip_path = raw_path.parent / "higgs_download.zip"
temp_raw_path = raw_path.with_name(raw_path.name + ".part")
try:
    import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)


## Step 2B — Add a ZIP validation helper

This small helper checks whether the downloaded ZIP has a readable directory and contains a `.csv.gz` file.

In [ ]:

def find_csv_gz_member(archive_path: Path):
    """Return the CSV.GZ member only when the ZIP directory is readable."""
    archive_path = Path(archive_path)
    if not archive_path.is_file():
        return None
    try:
        with zipfile.ZipFile(archive_path, "r") as archive:
            return next((name for name in archive.namelist() if name.lower().endswith(".csv.gz")), None)
    except (OSError, zipfile.BadZipFile, EOFError):
        return None


## Step 2C — Add retry and resume download logic

This defines the downloader only; the next cell actually starts the download. Interrupted bytes are kept, and HTTP Range resume is used when supported by the UCI server.

In [ ]:
import re
import time
import zipfile
from pathlib import Path


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _download_mode(response, existing, destination):
    if response.status_code == 416:
        Path(destination).unlink(missing_ok=True)
        raise IOError("The server rejected the resume offset; restarting from zero.")
    response.raise_for_status()
    append = existing > 0 and response.status_code == 206
    if append:
        content_range = response.headers.get("Content-Range", "")
        match = re.match(r"bytes\s+(\d+)-", content_range)
        if not match or int(match.group(1)) != existing:
            Path(destination).unlink(missing_ok=True)
            raise IOError("Server returned an unexpected byte range; restarting from zero.")
    return "ab" if append else "wb"


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _download_attempt(download_url, destination, session):
    destination = Path(destination)
    existing = destination.stat().st_size if destination.exists() else 0
    headers = {"Range": f"bytes={existing}-"} if existing else {}
    with session.get(download_url, headers=headers, stream=True, timeout=(30, 180)) as response:
        mode = _download_mode(response, existing, destination)
        expected = response.headers.get("Content-Length")
        received = 0
        with destination.open(mode) as target:
            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                if chunk:
                    target.write(chunk)
                    received += len(chunk)
        if expected is not None and received != int(expected):
            raise IOError(f"Incomplete HTTP response: received {received:,} of {int(expected):,} bytes.")
    if not find_csv_gz_member(destination):
        raise IOError("The downloaded ZIP is not complete yet.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def download_zip_resumable(download_url, destination, max_attempts=10):
    """Download with retries and HTTP Range resume where the server supports it."""
    import requests
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if find_csv_gz_member(destination):
        print("A readable ZIP is already present; reusing it.")
        return destination
    session = requests.Session()
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            _download_attempt(download_url, destination, session)
            print(f"ZIP received: {destination.stat().st_size / (1024**3):.2f} GiB.")
            return destination
        except Exception as exc:
            last_error = exc
            print(f"Download attempt {attempt}/{max_attempts} failed: {exc}")
            if attempt < max_attempts:
                time.sleep(min(2 ** (attempt - 1), 30))
    raise RuntimeError(f"Could not obtain a readable UCI ZIP. Rerun this cell. Last error: {last_error}")


## Step 2D — Download the complete archive

Run this cell to start or resume the complete UCI archive download. It skips downloading if the complete extracted CSV already exists or a valid archive is already saved.


In [ ]:
if raw_path.exists():
    print("Raw HIGGS data already exists; skipping download.")
elif find_csv_gz_member(zip_path):
    print("A readable ZIP archive already exists; ready to extract.")
else:
    print("Downloading the full HIGGS archive from UCI. This is a large file.")
    download_zip_resumable(url, zip_path, max_attempts=10)

if raw_path.exists():
    print("Raw file:", raw_path, f"({raw_path.stat().st_size / (1024**3):.2f} GiB)")
else:
    print("ZIP ready:", zip_path if find_csv_gz_member(zip_path) else "download incomplete; rerun this cell")


## Step 2E — Extract and validate the complete raw file

This cell extracts the complete `HIGGS.csv.gz` file. The model and EDA cells use this raw file directly after extraction.


In [ ]:
import gzip
import json
import shutil
import zipfile

expected_row_count = 11_000_000
validation_marker = raw_path.with_name(raw_path.name + ".validation.json")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def count_gzip_rows(path, chunk_size=4 * 1024 * 1024):
    """Fully read a gzip file and count CSV rows without loading it into memory."""
    import gzip
    newline_count = 0
    last_byte = b""
    with gzip.open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            newline_count += chunk.count(b"\n")
            last_byte = chunk[-1:]
    if last_byte and last_byte != b"\n":
        newline_count += 1
    return newline_count


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def write_validation_marker(path, rows):
    validation_marker.write_text(
        json.dumps({"rows": int(rows), "size_bytes": int(path.stat().st_size)}, indent=2),
        encoding="utf-8",
    )


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
raw_is_valid = False
if raw_path.is_file() and raw_path.stat().st_size > 0:
    try:
        marker = json.loads(validation_marker.read_text(encoding="utf-8"))
        with raw_path.open("rb") as raw_handle:
            raw_header = raw_handle.read(2)
        raw_is_valid = (
            marker.get("rows") == expected_row_count
            and marker.get("size_bytes") == raw_path.stat().st_size
            and raw_header == b"\x1f\x8b"
        )
    except (OSError, ValueError, TypeError):
        raw_is_valid = False


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
if raw_path.is_file() and raw_path.stat().st_size > 0 and not raw_is_valid:
    try:
        actual_rows = count_gzip_rows(raw_path)
        raw_is_valid = actual_rows == expected_row_count
        if raw_is_valid:
            write_validation_marker(raw_path, actual_rows)
        else:
            print(f"Existing raw file is incomplete ({actual_rows:,} rows); it will be replaced.")
    except Exception as exc:
        print("Existing raw file failed validation and will be replaced:", exc)
        raw_is_valid = False


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _extract_zip_member(zip_path, member_name, temp_path):
    temp_path.unlink(missing_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        with archive.open(member_name, "r") as source, temp_path.open("wb") as target:
            shutil.copyfileobj(source, target, length=4 * 1024 * 1024)
    return count_gzip_rows(temp_path)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def extract_and_validate_raw(expected_rows):
    member_name = find_csv_gz_member(zip_path)
    if not member_name:
        print("No valid ZIP member found; downloading or resuming the archive.")
        download_zip_resumable(url, zip_path, max_attempts=10)
        member_name = find_csv_gz_member(zip_path)
    if not member_name:
        raise RuntimeError("No readable .csv.gz member found; rerun the download step.")
    try:
        rows = _extract_zip_member(zip_path, member_name, temp_raw_path)
        if rows != expected_rows:
            raise ValueError(f"Expected {expected_rows:,} rows, found {rows:,}.")
        temp_raw_path.replace(raw_path)
        write_validation_marker(raw_path, rows)
    except Exception as exc:
        temp_raw_path.unlink(missing_ok=True)
        zip_path.unlink(missing_ok=True)
        raise RuntimeError("Extraction/gzip validation failed. Rerun the download step.") from exc
    zip_path.unlink(missing_ok=True)
    return rows


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
if raw_is_valid:
    complete_row_count = expected_row_count
    print("Raw HIGGS data is validated; skipping extraction.")
else:
    raw_path.unlink(missing_ok=True)
    validation_marker.unlink(missing_ok=True)
    complete_row_count = extract_and_validate_raw(expected_row_count)
if complete_row_count != expected_row_count:
    raise ValueError(f"Expected {expected_row_count:,} rows, found {complete_row_count:,}.")
print("Raw data ready:", raw_path)
print(f"Raw file size: {raw_path.stat().st_size / (1024**3):.2f} GiB")


## Step 2F — Confirm the complete dataset row count

This streaming check counts the rows in the extracted gzip file without loading the whole dataset into memory. Save this output for the data-card section of the report.


In [ ]:
if "complete_row_count" not in globals():
    complete_row_count = count_gzip_rows(raw_path)
if complete_row_count != 11_000_000:
    raise ValueError(f"Expected 11,000,000 rows, found {complete_row_count:,}.")
print("Complete raw file:", raw_path)
print("Complete row count:", f"{complete_row_count:,}")
print("Complete archive URL:", url)


## Step 3A — Use the complete training data and a smaller EDA/audit sample

The full raw HIGGS file is retained for model training. To avoid loading all 11 million rows into RAM for exploratory plots and audit checks, the notebook uses a configurable 250,000-row sample for EDA and reconciliation by default. Set `HIGGS_EDA_ROWS=11000000` only if your memory budget supports full-data EDA.

In [ ]:
import os
import sys
from pathlib import Path

root = Path(os.environ["HIGGS_REPO_ROOT"]).resolve()
src_dir = root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
expected_row_count = 11_000_000
if "complete_row_count" not in globals():
    raise RuntimeError("Run Step 2E and Step 2F before continuing.")
full_row_count = complete_row_count
if full_row_count != expected_row_count:
    raise ValueError(f"Expected {expected_row_count:,} rows, found {full_row_count:,}.")
training_sample_path = raw_path
if not raw_path.is_file():
    raise FileNotFoundError(f"Complete raw dataset is missing: {raw_path}. Finish Step 2 first.")
print("Complete training file:", raw_path)
print("Training rows:", f"{full_row_count:,}")


## Step 3B — Confirm the complete training input

The full raw file is already present, so no 1-million-row extraction is performed. The training process receives `data/raw/HIGGS.csv.gz` directly.


In [ ]:
print("Complete training input ready:", training_sample_path)
print("Training rows requested:", f"{full_row_count:,}")


### Step 3C — Prepare a memory-safe EDA and audit sample

The training job still uses all 11 million rows. This cell creates a separate, cached sample file for plotting and split/prediction reconciliation. The default is 250,000 rows; set `HIGGS_EDA_ROWS=11000000` before running setup only when full-data EDA is intentional.

In [ ]:
import gzip
import os
from pathlib import Path

eda_row_count = int(os.environ.get("HIGGS_EDA_ROWS", "250000"))
if not 10 <= eda_row_count <= full_row_count:
    raise ValueError(f"HIGGS_EDA_ROWS must be between 10 and {full_row_count:,}.")
processed_dir = root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
sample_path = processed_dir / f"higgs_eda_sample_{eda_row_count}.csv.gz"
sample_tmp_path = sample_path.with_name(sample_path.name + ".part")

def _sample_row_count(path):
    try:
        with gzip.open(path, "rt", encoding="utf-8") as handle:
            return sum(1 for _ in handle)
    except (OSError, EOFError):
        return -1


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
if _sample_row_count(sample_path) != eda_row_count:
    sample_path.unlink(missing_ok=True)
    sample_tmp_path.unlink(missing_ok=True)
    rows_written = 0
    try:
        with gzip.open(raw_path, "rt", encoding="utf-8") as source, gzip.open(sample_tmp_path, "wt", encoding="utf-8") as target:
            for line in source:
                if rows_written >= eda_row_count:
                    break
                target.write(line)
                rows_written += 1
        if rows_written != eda_row_count:
            raise ValueError(f"Expected {eda_row_count:,} sample rows, wrote {rows_written:,}.")
        if _sample_row_count(sample_tmp_path) != eda_row_count:
            raise IOError("The sample gzip file failed its row-count check.")
        sample_tmp_path.replace(sample_path)
    except Exception:
        sample_tmp_path.unlink(missing_ok=True)
        raise

print("Complete training input:", training_sample_path)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
print("Training rows requested:", f"{full_row_count:,}")
print("EDA/audit sample:", sample_path)
print("EDA rows requested:", f"{eda_row_count:,}")


## Step 3D — Generate saved EDA evidence

EDA runs on the cached, configurable sample created in Step 3C. This keeps plotting and correlation calculations memory-safe while leaving the training dataset unchanged.

### EDA fallback helper

The cloned public repository may not yet contain the optional report helper script. This local fallback creates the same essential report tables and figures directly in the notebook.


In [ ]:

def _save_eda_tables(X, y, eda_dir, json_module, pandas_module):
    stats = X.describe().T
    stats["missing_count"] = X.isna().sum()
    stats["signal_mean"] = X.loc[y == 1].mean()
    stats["background_mean"] = X.loc[y == 0].mean()
    stats.to_csv(eda_dir / "feature_summary.csv")
    btag = {name: sorted(float(value) for value in X[name].unique()) for name in X.columns if "btag" in name}
    (eda_dir / "btag_summary.json").write_text(json_module.dumps(btag, indent=2), encoding="utf-8")
    corr = X.corr()
    corr.to_csv(eda_dir / "correlation_matrix.csv")
    means = pandas_module.DataFrame({"signal_mean": X.loc[y == 1].mean(), "background_mean": X.loc[y == 0].mean()})
    means.rename_axis("feature").reset_index().to_csv(eda_dir / "class_conditional_means.csv", index=False)
    balance = pandas_module.Series(y).value_counts().sort_index()
    balance.rename_axis("class_label").reset_index(name="events").to_csv(eda_dir / "class_balance.csv", index=False)
    return corr


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _save_eda_figures(y, corr, eda_dir, plt, np):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Background (0)", "Signal (1)"], [int((y == 0).sum()), int((y == 1).sum())], color=["#5f6b7a", "#24558f"])
    ax.set_ylabel("Events")
    ax.set_title("Class balance in the EDA sample")
    fig.tight_layout()
    fig.savefig(eda_dir / "class_balance.png", dpi=160)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(9, 7))
    image = ax.imshow(corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    ax.set_title("Pearson correlation matrix")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(eda_dir / "correlation_heatmap.png", dpi=160)
    plt.close(fig)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def generate_eda_fallback(sample_path, eda_dir):
    import json
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from higgs_event_lab.data import load_higgs_sample

    batch = load_higgs_sample(sample_path, n_rows=eda_row_count, feature_group="all")
    X, y = batch.X, batch.y.to_numpy()
    eda_dir.mkdir(parents=True, exist_ok=True)
    corr = _save_eda_tables(X, y, eda_dir, json, pd)
    _save_eda_figures(y, corr, eda_dir, plt, np)
    summary = {
        "rows": int(len(X)), "feature_count": int(X.shape[1]),
        "signal_fraction": float((y == 1).mean()),
        "missing_values": int(X.isna().sum().sum()),
        "figures": ["class_balance.png", "correlation_heatmap.png"],
    }
    (eda_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("EDA fallback completed:", eda_dir)


### Run the EDA step

Use the saved-script path when the cloned repository includes it; otherwise the fallback helper above runs the analysis directly and creates the report artifacts.


In [ ]:
eda_script = root / "scripts" / "run_eda.py"
eda_result = None
if eda_script.is_file():
    eda_result = subprocess.run([sys.executable, str(eda_script), "--data", str(sample_path), "--rows", str(eda_row_count), "--output-dir", str(root / "outputs" / "eda")], cwd=root, check=False)
if eda_result is None or eda_result.returncode != 0:
    generate_eda_fallback(sample_path, root / "outputs" / "eda")
else:
    print("EDA exit code:", eda_result.returncode)


## Step 3E — Reconcile the sample and saved full-data predictions

The audit uses the smaller sample for partition checks and reads the full raw file once for row-count/hash evidence. Prediction files from the full model run are discovered in `outputs/*_test_predictions.csv`.

In [ ]:

def _fallback_partition_report(sample_path):
    import pandas as pd
    from higgs_event_lab.features import ALL_COLUMNS
    from higgs_event_lab.protocol import SplitConfig, create_stratified_splits
    frame = pd.read_csv(sample_path, header=None, names=list(ALL_COLUMNS))
    labels = frame["class_label"].to_numpy(dtype="int64")
    split = create_stratified_splits(labels, SplitConfig(seed=42))
    partitions = {"train": split.train, "validation": split.validation, "test": split.test}
    hashes = pd.util.hash_pandas_object(frame.drop(columns=["class_label"]), index=False).to_numpy()
    hash_sets = {name: set(hashes[indexes].tolist()) for name, indexes in partitions.items()}
    pairs = [("train", "validation"), ("train", "test"), ("validation", "test")]
    overlap = sum(len(hash_sets[left] & hash_sets[right]) for left, right in pairs)
    return frame, split, int(overlap)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _fallback_prediction_report(root):
    import pandas as pd
    paths = sorted((root / "outputs").glob("*_test_predictions.csv"))
    sets = [set(pd.read_csv(path)["row_index"].astype(int)) for path in paths]
    common = len(set.intersection(*sets)) if sets else 0
    return paths, int(common)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _fallback_raw_report(raw_path):
    import gzip
    import hashlib
    digest = hashlib.sha256()
    rows = 0
    with gzip.open(raw_path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
            rows += chunk.count(b"\n")
    return rows, digest.hexdigest()


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

audit_script = root / "scripts" / "reconcile_results.py"
audit_result = None
if audit_script.is_file():
    audit_result = subprocess.run(
        [sys.executable, str(audit_script), "--sample", str(sample_path), "--raw", str(raw_path),
         "--output", str(root / "outputs" / "reconciliation_report.json"),
         "--prediction-glob", "outputs/*_test_predictions.csv"],
        cwd=root, check=False,
    )


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
if audit_result is None or audit_result.returncode != 0:
    import json
    frame, split, duplicate_overlap = _fallback_partition_report(sample_path)
    prediction_paths, common_test_indices = _fallback_prediction_report(root)
    raw_rows, raw_sha256 = _fallback_raw_report(raw_path)
    report = {
        "sample": {"rows": int(len(frame)), "columns": int(frame.shape[1])},
        "split": split.metadata(),
        "partition_audit": {"duplicate_feature_rows_across_partitions": duplicate_overlap,
                             "partitions_disjoint": duplicate_overlap == 0},
        "prediction_audit": {"files_checked": len(prediction_paths),
                              "shared_test_index_count": common_test_indices},
        "raw_file": {"rows": raw_rows, "sha256": raw_sha256,
                     "size_bytes": raw_path.stat().st_size},
    }
    audit_path = root / "outputs" / "reconciliation_report.json"
    audit_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("Reconciliation fallback completed:", audit_path)
else:
    print("Reconciliation exit code:", audit_result.returncode)


## Research and authorship

Saumya Mishra owns the physics background, dataset guide, and QA plan that explain why the 21 low-level and 7 high-level features are the right problem here. CosmicMagnetar tuned the models and prepared this Colab walkthrough so the numbers can be reproduced without claiming physics the benchmark never measures. Use this note in your report and viva so the reader understands who wrote what.

## Step 4A — Check training inputs and load the script

This prepares the variables used by the compatibility checks. It does not start training. If the cloned script already contains this definition, the cell leaves it unchanged and remains safe to rerun.


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab")).resolve()
script = root / "scripts" / "train_full_colab.py"
training_sample_path = root / "data" / "raw" / "HIGGS.csv.gz"

if not script.is_file():
    raise FileNotFoundError(f"Training script not found: {script}. Run Step 1 first.")
if not training_sample_path.is_file():
    raise FileNotFoundError(f"Training sample not found: {training_sample_path}. Run Step 3B first.")

original_source = script.read_text(encoding="utf-8")
script_source = original_source
print("Training script:", script)
print("Training sample:", training_sample_path)


## Step 4B — Fix `OUTPUTS_DIR` if it is missing

This addresses the previous `NameError: OUTPUTS_DIR is not defined` error. Run in order after Step 4A. This compatibility step keeps older repository copies runnable while preserving the script syntax before training begins.


In [ ]:

def ensure_outputs_dir_defined(script_source):
    header = script_source.split("def evaluate_run", 1)[0]
    if not re.search(r"(?m)^OUTPUTS_DIR\s*=", header):
        root_line = re.search(r"(?m)^ROOT_DIR\s*=.*$", header)
        if not root_line:
            raise RuntimeError("Cannot patch OUTPUTS_DIR: ROOT_DIR definition was not found.")
        pos = root_line.end()
        script_source = script_source[:pos] + '\nOUTPUTS_DIR = ROOT_DIR / "outputs"\nOUTPUTS_DIR.mkdir(parents=True, exist_ok=True)' + script_source[pos:]
    else:
        header = script_source.split("def evaluate_run", 1)[0]
        if not re.search(r"(?m)^OUTPUTS_DIR\.mkdir\(", header):
            assign = re.search(r"(?m)^OUTPUTS_DIR\s*=.*$", header)
            if assign:
                script_source = script_source[:assign.end()] + "\nOUTPUTS_DIR.mkdir(parents=True, exist_ok=True)" + script_source[assign.end():]
    return script_source


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
script_source = ensure_outputs_dir_defined(script_source)
print("OUTPUTS_DIR compatibility check complete.")


## Step 4C — Fix missing metric imports and save the patch

This addresses the previous `NameError: roc_auc_score is not defined` error. The training script is changed only if a fix is needed.

In [ ]:

def ensure_metric_imports(script_source):
    metrics_import = "from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score"
    header = script_source.split("def evaluate_run", 1)[0]
    if metrics_import in header:
        return script_source
    pandas_import = re.search(r"(?m)^import pandas as pd\s*$", header)
    if pandas_import:
        pos = pandas_import.end()
        return script_source[:pos] + "\n" + metrics_import + script_source[pos:]
    first_function = re.search(r"(?m)^def evaluate_run\b", script_source)
    if not first_function:
        raise RuntimeError("Could not safely patch metric imports in the training script.")
    return script_source[:first_function.start()] + metrics_import + "\n\n" + script_source[first_function.start():]


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def ensure_training_memory_cleanup(script_source):
    if "import gc" not in script_source:
        script_source = script_source.replace("import argparse\n", "import argparse\nimport gc\n", 1)
    if "    features = df.drop(columns=[TARGET_COLUMN])\n    del df\n    gc.collect()" not in script_source:
        script_source = script_source.replace(
            "    features = df.drop(columns=[TARGET_COLUMN])\n",
            "    features = df.drop(columns=[TARGET_COLUMN])\n    del df\n    gc.collect()\n", 1,
        )
    cleanup = (
        "    # Release scaled arrays before MLP allocates float32 arrays.\n"
        "    del X_train, X_val, X_test\n    gc.collect()\n"
        "    mlp_result = train_mlp(features, labels, split=split, config=mlp_config)\n"
    )
    if "del X_train, X_val, X_test" not in script_source:
        script_source = script_source.replace(
            "    mlp_result = train_mlp(features, labels, split=split, config=mlp_config)\n",
            cleanup, 1,
        )
    return script_source


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def ensure_summary_row_count(script_source):
    if '    row_count = len(df)\n' not in script_source:
        old = '    df = pd.read_csv(sample_path, header=None, names=list(ALL_COLUMNS))\n    print(f"Loaded {len(df):,} rows from {sample_path.name}")\n'
        new = '    df = pd.read_csv(sample_path, header=None, names=list(ALL_COLUMNS))\n    row_count = len(df)\n    print(f"Loaded {row_count:,} rows from {sample_path.name}")\n'
        script_source = script_source.replace(old, new, 1)
    return script_source.replace('"rows": len(df),', '"rows": row_count,')


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
compile(script_source, str(script), "exec")
if script_source != original_source:
    script.write_text(script_source, encoding="utf-8")
    print("Saved compatible training script:", script)
else:
    print("Training script already matches the current repository version.")


## Step 4D — Train with visible progress

The default is a resource-safe training run on the prepared analysis sample (normally 250,000 rows). This is a real trained/evaluated run, but **it is not an 11-million-row benchmark**. Logs are printed before data loading and between model stages, per-epoch MLP progress is shown, and a heartbeat appears every 10 seconds while waiting for output.

In code cell 42, change `RUN_FULL_TRAINING = False` to `True` to attempt the full 11-million-row run. The full run uses the original model settings and can be very slow or run out of RAM on a free runtime.


In [ ]:
env = os.environ.copy()
env["HIGGS_REPO_ROOT"] = str(root)
env["HIGGS_SAMPLE_PATH"] = str(training_sample_path)
env["PYTHONUNBUFFERED"] = "1"

print("Project root:", root)
print("Training input:", training_sample_path)
print("Starting full-data training...")
result = subprocess.run(
    [sys.executable, "-u", str(script), "--sample", str(training_sample_path)],
    cwd=root,
    env=env,
    check=False,
)
if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}.")
print("Training process finished successfully.")


## Step 4E — Confirm the results file exists

This is a quick checkpoint before displaying the metrics. This checkpoint prevents the results-display cells from hiding a failed or incomplete training process.


In [ ]:
summary_path = root / "outputs" / "full_colab_results.json"
if not summary_path.is_file():
    raise FileNotFoundError(f"Training finished but summary was not created: {summary_path}")
print("Training summary found:", summary_path)


## Step 5A — Load the saved results

Run after training completes. These values are the report-ready test metrics and should be copied only from the saved JSON summary.


In [ ]:
import json
from pathlib import Path

summary_path = root / "outputs" / "full_colab_results.json"
if not summary_path.exists():
    raise FileNotFoundError("No training summary found. Run Step 4D first.")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("Sample rows:", summary["rows"])
print("Split:", summary["split"])


## Step 5B — Display model metrics The listed files are the evidence package for the report, dashboard and later reproducibility review. The table is the report-ready comparison of held-out performance and runtime for the three Protocol B models.


In [ ]:
print(f"{'Model':<28} {'ROC-AUC':>8} {'PR-AUC':>8} {'Brier':>8} {'Accuracy':>9} {'Seconds':>9}")
print("-" * 80)
for record in summary["runs"]:
    print(
        f"{record['model']:<28} "
        f"{record['test_roc_auc']:>8.4f} "
        f"{record['test_pr_auc']:>8.4f} "
        f"{record['brier_score']:>8.4f} "
        f"{record['test_accuracy']:>9.4f} "
        f"{record['training_seconds']:>9.1f}"
    )


## Step 5C — List saved artifacts This inventory lets the team attach the generated JSON, prediction tables and model files as reproducibility evidence. Confirm that these paths exist before downloading the results from the temporary cloud runtime.


In [ ]:
print("Artifacts:")
for artifact in summary.get("artifacts", []):
    print(" ", artifact)
